# Four-Class Object Image Classification with an Artificial Neural Network

**Project type:** Computer Vision · Supervised Learning · Artificial Neural Networks

## Project objective
This notebook builds a feed-forward artificial neural network (ANN) that classifies an input image into one of four folder-based categories: **ice cream cone, digit, cricket ball, or car**.

The workflow covers image loading, preprocessing, stratified train/validation/test splits, model training, evaluation, and prediction examples. The model uses flattened grayscale pixels as input; it is a simple baseline rather than a state-of-the-art computer-vision model.

> **Important:** The dataset is not included in this repository. Add it locally using the folder structure shown below. Results will be generated when you run the notebook on your own dataset.

## 1. Dataset folder structure

Place your dataset in a folder named `Dataset` next to this notebook's project root. Each class must have its own subfolder, with image files inside:

```text
project-root/
├── Dataset/
│   ├── car/
│   ├── cricket ball/
│   ├── digit/
│   └── ice cream cone/
├── notebooks/
│   └── artificial-neural-network-classification.ipynb
├── README.md
└── requirements.txt
```

Folder names may contain spaces. The notebook reads class names from the folder names and sorts them to keep label mapping consistent. Do not upload the dataset or private images to GitHub.

## 2. Install and import dependencies

Install packages from `requirements.txt` in your Python environment before running the notebook.

In [ ]:
import os
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

## 3. Set reproducibility and dataset path

A fixed seed makes the data split and training more repeatable, although exact results can still vary across hardware and TensorFlow versions.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMAGE_SIZE = (100, 100)
dataset_candidates = [Path("Dataset"), Path("../Dataset")]
DATASET_DIR = next((candidate for candidate in dataset_candidates if candidate.is_dir()), None)

if DATASET_DIR is None:
    raise FileNotFoundError(
        "Dataset folder not found. Create Dataset/ in the project root "
        "and add one subfolder per class."
    )

class_names = sorted(
    p.name for p in DATASET_DIR.iterdir() if p.is_dir()
)

if len(class_names) != 4:
    raise ValueError(
        f"Expected 4 class folders, but found {len(class_names)}: {class_names}"
    )

print("Classes:", class_names)

## 4. Load and preprocess images

Images are loaded as grayscale, resized to 100 × 100 pixels, and normalized to the range 0–1. Each image is flattened into a 10,000-value vector because a standard dense ANN expects a one-dimensional feature vector.

This is intentionally a baseline approach. Flattening removes the explicit 2D spatial structure that convolutional neural networks can learn.

In [ ]:
valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
images = []
labels = []
skipped_files = []

for label_id, class_name in enumerate(class_names):
    class_dir = DATASET_DIR / class_name

    for image_path in sorted(class_dir.iterdir()):
        if not image_path.is_file() or image_path.suffix.lower() not in valid_extensions:
            continue

        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            skipped_files.append(str(image_path))
            continue

        image = cv2.resize(image, IMAGE_SIZE, interpolation=cv2.INTER_AREA)
        images.append(image)
        labels.append(label_id)

if not images:
    raise ValueError("No readable images were found. Check your Dataset/ folders.")

X = np.asarray(images, dtype=np.float32) / 255.0
y = np.asarray(labels, dtype=np.int64)

# Dense ANN input: one flattened grayscale image per row
X = X.reshape(len(X), -1)

print(f"Images loaded: {len(X)}")
print(f"Features per image: {X.shape[1]}")
print(f"Unreadable images skipped: {len(skipped_files)}")

## 5. Inspect class balance and sample images

Check the number of images per class before training. Strong class imbalance can make accuracy alone misleading.

In [ ]:
class_counts = pd.Series(
    [class_names[label] for label in y]
).value_counts().reindex(class_names, fill_value=0)

display(class_counts.rename("image_count").to_frame())

ax = class_counts.plot(kind="bar", figsize=(8, 4))
ax.set_title("Images per class")
ax.set_xlabel("Class")
ax.set_ylabel("Number of images")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
rng = np.random.default_rng(SEED)
sample_indices = rng.choice(len(X), size=min(12, len(X)), replace=False)

fig, axes = plt.subplots(3, 4, figsize=(10, 7))
for ax, idx in zip(axes.flat, sample_indices):
    ax.imshow(X[idx].reshape(IMAGE_SIZE), cmap="gray")
    ax.set_title(class_names[y[idx]])
    ax.axis("off")

for ax in axes.flat[len(sample_indices):]:
    ax.axis("off")

fig.suptitle("Sample images from the dataset")
plt.tight_layout()
plt.show()

## 6. Create train, validation, and test sets

The split is approximately **70% training, 15% validation, and 15% test**, using stratification to preserve class proportions where the class counts allow it. The test set is held back for the final evaluation.

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=SEED,
    stratify=y,
)

validation_fraction_of_train_val = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=validation_fraction_of_train_val,
    random_state=SEED,
    stratify=y_train_val,
)

split_summary = pd.DataFrame({
    "split": ["Train", "Validation", "Test"],
    "images": [len(X_train), len(X_val), len(X_test)],
})
split_summary["percentage"] = (
    split_summary["images"] / len(X) * 100
).round(1)
display(split_summary)

print("Input shape:", X_train.shape[1])

## 7. Build the ANN

The network uses two ReLU hidden layers and dropout regularization, followed by a four-unit softmax output layer. The output layer returns a probability for each class.

In [ ]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(256, activation="relu"),
    Dropout(0.30),
    Dense(128, activation="relu"),
    Dropout(0.20),
    Dense(len(class_names), activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## 8. Train the model

Early stopping monitors validation loss and restores the weights from the best validation epoch. This can reduce unnecessary training and help limit overfitting.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True,
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1,
)

## 9. Review training curves

A large gap between training and validation performance may indicate overfitting. Curves should be interpreted alongside the final test metrics, not as proof of generalization by themselves.

In [ ]:
history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["accuracy"], label="Train")
axes[0].plot(history_df["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy by epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history_df["loss"], label="Train")
axes[1].plot(history_df["val_loss"], label="Validation")
axes[1].set_title("Loss by epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Evaluate on the held-out test set

The classification report provides precision, recall, and F1-score for each class. Review these alongside overall accuracy, especially if the class distribution is uneven.

In [ ]:
test_probabilities = model.predict(X_test, verbose=0)
y_pred = np.argmax(test_probabilities, axis=1)

test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {test_accuracy:.3f} ({test_accuracy * 100:.2f}%)")
print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    zero_division=0,
))

## 11. Confusion matrix

Rows represent the actual class and columns represent the predicted class. Off-diagonal values show the kinds of mistakes made by the model.

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(len(class_names)),
)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.title("Test-set confusion matrix")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 12. Inspect individual predictions

The confidence shown here is the model's largest softmax output. It is not a calibrated probability that the prediction is correct.

In [ ]:
num_examples = min(12, len(X_test))
fig, axes = plt.subplots(3, 4, figsize=(11, 8))

for position, ax in enumerate(axes.flat):
    if position >= num_examples:
        ax.axis("off")
        continue

    image = X_test[position].reshape(IMAGE_SIZE)
    actual_name = class_names[y_test[position]]
    predicted_name = class_names[y_pred[position]]
    confidence = float(np.max(test_probabilities[position]))

    ax.imshow(image, cmap="gray")
    ax.set_title(
        f"Actual: {actual_name}\nPredicted: {predicted_name}\nScore: {confidence:.2f}",
        fontsize=8,
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

## 13. Save the trained model (optional)

Run this cell if you want to save the model locally. The generated model file is excluded from GitHub by the provided `.gitignore`; large model files generally should not be committed to a source-code repository.

In [ ]:
model.save("ann_object_classifier.keras")
print("Saved model to ann_object_classifier.keras")

## 14. Conclusions and limitations

After running the notebook, write a short summary using your actual results:
- Which class was easiest and hardest to classify?
- Which mistakes were most common in the confusion matrix?
- Did training and validation curves suggest overfitting?
- How might a CNN compare with this dense ANN?

### Limitations
- The model is a baseline and uses flattened grayscale pixels rather than learned spatial features.
- Results depend on dataset size, image quality, class balance, and how the images were collected.
- A random split may not measure performance on images from a different source or environment.
- Softmax scores are not calibrated confidence estimates.
- This is an educational image-classification experiment, not a general-purpose object detector.

## Reproducibility notes
Record the dataset source, date accessed, image counts per class, package versions, and any changes you make to preprocessing or model settings. Do not report a metric until you have run the notebook and confirmed the corresponding output.